In [ ]:
%pip install ipython-sql psycopg2 pandas matplotlib numpy matplotlib

In [ ]:
%load_ext sql

In [ ]:
from urllib.parse import quote_plus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
DB_URL = "postgresql://intern_read_only:%s@api.algotrade.vn:5432/algotradeDB" % quote_plus("Bingo@0711")
%sql $DB_URL

In [ ]:
def query_command(symbol, start_date, end_date):
    return """
    select tb_max.datetime as datetime, tb_max.tickersymbol as symbol,
    tb_max.price as max,
    tb_min.price as min,
    tb_close.price as close,
    tb_open.price as open,
    tb_total.quantity as total_quantity
    from (
        select *
        from quote.max m
    ) tb_max
    inner join (
        select *
        from quote.min m
    ) tb_min
    on tb_max.tickersymbol = tb_min.tickersymbol
    and tb_max.datetime = tb_min.datetime
    inner join (
        select *
        from quote.close m
    ) tb_close
    on tb_min.tickersymbol = tb_close.tickersymbol
    and tb_min.datetime = tb_close.datetime
    inner join (
        select *
        from quote.open m
    ) tb_open
    on tb_open.tickersymbol = tb_close.tickersymbol
    and tb_open.datetime = tb_close.datetime
    inner join (
        select TO_CHAR(datetime , 'YYYY-MM-DD') as datetime , m.tickersymbol,
        max(m.quantity) as quantity
        from quote.total m
        where m.tickersymbol = '{symbol}'
        and m.datetime between '{start_date}' and '{end_date}'
        group by TO_CHAR(datetime , 'YYYY-MM-DD'), m.tickersymbol
    ) tb_total
    on tb_open.tickersymbol = tb_total.tickersymbol
    and tb_open.datetime = to_date(tb_total.datetime, 'YYYY-MM-DD')
    where tb_max.tickersymbol = '{symbol}'
    and tb_max.datetime between '{start_date}' and '{end_date}';
""".format(symbol=symbol, start_date=start_date, end_date=end_date)

In [ ]:
df = pd.DataFrame()

query_var = [
    ('VN30F2310', '2023-09-22', '2023-10-21'),
    ('VN30F2311', '2023-10-21', '2023-11-17'),
    ('VN30F2312', '2023-11-17', '2023-12-22'),
    ('VN30F2401', '2023-12-22', '2024-01-19'),
    ('VN30F2402', '2024-01-19', '2024-02-16')
]

for symbol, start_date, end_date in query_var:
    sql_query = query_command(symbol, start_date, end_date)
    result = %sql $sql_query
    df = pd.concat([df, result.DataFrame()], ignore_index=True)

In [ ]:
# Convert data type
df['datetime'] = pd.to_datetime(df['datetime'])
df['open'] = df['open'].astype(float)
df['close'] = df['close'].astype(float)
df['max'] = df['max'].astype(float)
df['min'] = df['min'].astype(float)

In [ ]:
# Plot closing price
df.plot(x='datetime', y='close', kind='line', figsize=(8, 4), title="In-sample data - Close price", xlabel="Date", ylabel="Close price", legend=False)

In [ ]:
# Plot max price
df.plot(x='datetime', y='max', kind='line', figsize=(8, 4), title="In-sample data - Max price", xlabel="Date", legend=False)

In [ ]:
# Plot min price
df.plot(x='datetime', y='min', kind='line', figsize=(8, 4), title="In-sample data - Min price", xlabel="Date", legend=False)

In [ ]:
# Calculate typical price for each day
df['typical_price'] = (df['max'] + df['min'] + df['close']) / 3

In [ ]:
# set number of days and standard deviations to use for Bollinger Bands
window = 10
no_of_std = 1.9

rolling_mean = df['typical_price'].rolling(window).mean()
rolling_std = df['typical_price'].rolling(window).std()

In [ ]:
# Calculate three Bollinger Bands
df['Middle Band'] = rolling_mean
df['Upper Band'] = rolling_mean + (rolling_std * no_of_std)
df['Lower Band'] = rolling_mean - (rolling_std * no_of_std)

In [ ]:
# Plot the Bollinger Bands
df[['datetime', 'Middle Band', 'Upper Band', 'Lower Band']].plot(x='datetime', y=['Middle Band', 'Upper Band', 'Lower Band'], kind='line', figsize=(8, 4), title="Bollinger Bands", xlabel="Date", legend=True)